In [ ]:
from rocqiomics import Rocqiomics
from rocqiomics.personal_utils import prepare_data_dicts

from monai.transforms import (
    Compose,
    ScaleIntensityd,
    NormalizeIntensityd,
    Spacingd,
)

BASEPATH = r"C:\Users\ri54995\AppData\Local\Programs\Python\envs\radiomics\Lib\site-packages\rocqiomics\test_data"

[[23,
  24,
  25,
  26,
  27,
  28,
  29,
  30,
  31,
  32,
  33,
  34,
  35,
  36,
  37,
  38,
  39,
  40,
  41,
  42,
  43,
  44,
  45,
  46,
  47,
  48,
  49,
  50,
  51,
  52,
  53,
  54,
  55,
  56,
  57,
  58,
  59,
  60,
  61,
  62,
  63,
  64,
  65,
  66,
  67,
  68,
  69,
  70,
  71,
  72,
  73,
  74,
  75,
  76,
  77,
  78,
  79,
  80,
  81,
  82,
  83,
  84,
  85,
  86,
  87,
  88,
  89,
  90,
  91,
  92,
  93,
  94,
  95,
  96,
  97,
  98,
  99,
  100,
  101,
  102,
  103,
  104,
  105,
  106,
  107,
  108,
  109,
  110,
  111,
  112,
  113,
  114,
  115,
  116,
  117,
  118,
  119,
  120,
  121,
  122,
  123,
  124,
  125,
  126,
  127,
  128,
  129,
  130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  146,
  147,
  148,
  149,
  150,
  151,
  152,
  153,
  154,
  155,
  156,
  157,
  158,
  159,
  160,
  161,
  162,
  163,
  164,
  165,
  166,
  167,
  168,
  169,
  170,
  171,
  172,
  173,
  174,
  175,
  176,

# Prepare `data_dicts`

`data_dicts` is the main input to the pipeline, and should minimally contain `image` and `mask` keywords for each case.
Additional metadata such as timepoint, modality, etc can be added via the `metadata` keyword.

In [19]:
# Custom function that generates data dicts from my usual folder structure.
# You don't need to use this, of course, as long as you generate the data_dicts with the required structure.
data_dicts, missing_img, missing_seg = prepare_data_dicts(
    base_dirpath=BASEPATH,
    mask_names=['Snickers2D'],
    modalities=['T2'],
    timepoints=['d1'],
)

data_dicts[0]

{'case_id': 'RGS021425_ph1',
 'image': 'C:\\Users\\ri54995\\AppData\\Local\\Programs\\Python\\envs\\radiomics\\Lib\\site-packages\\rocqiomics\\test_data\\Images\\d1_T2_RGS021425_ph1.nrrd',
 'mask': 'C:\\Users\\ri54995\\AppData\\Local\\Programs\\Python\\envs\\radiomics\\Lib\\site-packages\\rocqiomics\\test_data\\Masks\\Snickers2D\\d1_T2_RGS021425_ph1.seg.nrrd',
 'metadata': {'mask_name': 'Snickers2D', 'timepoint': 'd1', 'modality': 'T2'}}

# Run the pipeline

In [20]:
pipeline = Rocqiomics(
    data_dicts=data_dicts,
    preprocessing=Compose([
        NormalizeIntensityd(keys=['image']), # Standardize image intensity (subtract mean and divide by standard dev)
        ScaleIntensityd(keys=['image'], factor=99.0, minv=None, maxv=None), # Scale images by factor of 100 (99 + 1)
        Spacingd(keys=['image', 'mask'], pixdim=(0.125, 0.125, 1.0), mode=[3, 'nearest']), # Resample image and mask to specified voxel size
    ]),
    bin_width=1.0,
    save_results=False,
    save_dirpath='tutorial_results',
    save_by_columns=['mask_name', 'modality', 'timepoint'], # Save results to different files based on mask name, modality, and timepoint
    case_limit=1,
    force_2D=True,
    force_2D_dimension=0,
)
pipeline.run_pipeline()
results = pipeline.get_results()

Rocqiomics INFO:	 Validating images and masks...


Rocqiomics INFO:	 Pipeline Initialized | Cases: 1 | Excluded cases: 0
Rocqiomics INFO:	 Case 0/0 done in 11.59s	case_id: RGS021425_ph1	mask_name: Snickers2D	timepoint: d1	modality: T2	augmentation: aug_0


In [21]:
results.head(6)

,diagnostics_Versions_PyRadiomics,diagnostics_Versions_Numpy,diagnostics_Versions_SimpleITK,diagnostics_Versions_PyWavelet,diagnostics_Versions_Python,diagnostics_Configuration_Settings,diagnostics_Configuration_EnabledImageTypes,diagnostics_Image-original_Hash,diagnostics_Image-original_Dimensionality,diagnostics_Image-original_Spacing,...,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength,mask_name,timepoint,modality,augmentation
case_id,,,,,,,,,,,,,,,,,,,,,
RGS021425_ph1,v3.1.0,1.26.4,2.5.2,1.6.0,3.9.13,"{'minimumROIDimensions': 2, 'minimumROISize': ...",{'Original': {}},d2905c171f37ce8240e980e773329736370e5007,3D,"(0.125, 0.125, 1.0)",...,0.12698822555354655,0.03397840553658922,0.000840307876245455,796807.5758077905,1.472927638946014,95.53280281779453,Snickers2D,d1,T2,aug_0
